In [1]:
# pip install keras-tuner

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import L1, L2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD, Adam

from keras_tuner import RandomSearch

I0000 00:00:1795060464.392072    7347 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795060464.392663    7347 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795060464.437352    7347 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1795060465.532992    7347 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795060465.533369    7347 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
df = pd.read_csv("titanic.csv")

In [4]:
# Cabin is missing for 687 of 891 passengers, so dropping rows with any NaN
# kept only 183 rows. Drop Cabin (and the PassengerId row ID) and impute instead.
df = df.drop(columns=['Cabin', 'PassengerId'])
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [5]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_7347/3006716147.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(inplace=True)


In [6]:
X = df.drop(['Survived', 'Name', 'Ticket'], axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8,  random_state=42)

In [7]:
cat_cols = list(X_train.select_dtypes(include=['object']).columns)
encoder = OrdinalEncoder()
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols] )
X_test[cat_cols] = encoder.transform(X_test[cat_cols] )

/tmp/ipykernel_7347/294707101.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = list(X_train.select_dtypes(include=['object']).columns)


In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
X_train.shape[1]

7

In [10]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(64, activation='relu', kernel_regularizer=L1(0.01)),
    Dropout(0.7),
    Dense(32, activation='relu', kernel_regularizer=L2(0.001)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.summary()

/home/user/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1795060465.817606    7347 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,553 (60.75 KB)

 Trainable params: 15,553 (60.75 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile( optimizer=SGD(momentum=0.9),
    # optimizer=Adam(learning_rate=0.0001),
    loss = 'binary_crossentropy', metrics = ['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=5)

model.fit(X_train, y_train, epochs=20,
          validation_split=0.1, batch_size=32, 
          callbacks=[early_stopping])

Epoch 1/20


 1/20 ━━━━━━━━━━━━━━━━━━━━ 12s 675ms/step - accuracy: 0.3750 - loss: 5.2098

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5422 - loss: 4.9945 - val_accuracy: 0.6944 - val_loss: 4.6383


Epoch 2/20


 1/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5000 - loss: 4.7072

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6141 - loss: 4.3246 - val_accuracy: 0.6944 - val_loss: 3.9118


... (50 outputs left out)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7656 - loss: 0.6054 - val_accuracy: 0.8611 - val_loss: 0.4365


Epoch 20/20


 1/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7812 - loss: 0.5957

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7812 - loss: 0.5845 - val_accuracy: 0.8750 - val_loss: 0.4270


In [12]:
model.evaluate(X_test, y_test)

1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8438 - loss: 0.5241

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7989 - loss: 0.5245 


[0.5245485305786133, 0.7988826632499695]

## hyperparams tune

In [13]:
# funcation to find best params
def build_model(hp):
    model = Sequential([
        # hp.Int("units", min_value=32, max_value=256, step=32)
        # units the key
        # min_value = the min value to start with
        # max_value = the max value to end with
        # step is the gap between any choice
            Dense(units=hp.Int("units", min_value=32, max_value=256, step=32), 
                  activation='relu', input_shape=(X_train.shape[1],)),
            
            # hp.Choice('activation', ['relu', 'sigmoid'])
            # choice either relu or sigmoid
            Dense(10, activation=hp.Choice('activation', ['relu', 'sigmoid'])),

            Dense(1, activation="sigmoid")
    ])

    model.compile(
        # learning_rate=hp.Choice('learning_rate', [0.01, 0.001, 0.0001]
        # find any learning rate in list: [0.01, 0.001, 0.0001]
        optimizer=Adam(learning_rate=hp.Choice('learning_rate', [0.01, 0.001, 0.0001])),
        loss = 'binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# find random param
tuner = RandomSearch(
    build_model,  # model arch
    objective='val_loss',  # moniter to check the best model (val_accuracy)
    max_trials=3,  # number of make experiments
    directory = 'lab4',  # dir to save each experiment
    project_name = 'session 6' # folder for this project
)

early_stopping = EarlyStopping(patience=5, 
                               restore_best_weights=True)

checkpoints = ModelCheckpoint("best_model.keras", 
                              monitor="val_loss", 
                              save_best_only=True)

tuner.search(X_train, y_train, 
             epochs=5, 
             validation_split=0.1,
             callbacks=[early_stopping, checkpoints])

Trial 3 Complete [00h 00m 02s]
val_loss: 0.5229381918907166

Best val_loss So Far: 0.5229381918907166
Total elapsed time: 00h 00m 18s


In [14]:
# get best model with best params
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/home/user/venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(store)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,173 (4.58 KB)

 Trainable params: 1,173 (4.58 KB)

 Non-trainable params: 0 (0.00 B)